In [ ]:
# Colab setup -- installs SoftMobility when running on Google Colab.
# Tip: For large simulations, switch to a GPU runtime first:
#   Runtime → Change runtime type → T4 GPU (free)
# Then re-run this cell to install the CUDA-enabled JAX.
try:
    import google.colab  # noqa: F401
    import shutil

    if shutil.which("nvidia-smi"):
        %pip install -q -U softmobility "jax[cuda12]"
    else:
        %pip install -q softmobility
except ImportError:
    pass

# Tutorial 05. Figure styling 

SoftMobility ships a small matplotlib styling helper, `softmobility.classes.figstyle`, that applies the paper aesthetic (white background, black box, no grid, Helvetica labels at 11 pt, fixed colour palette, orthographic 3D camera) used in every other tutorial.

This notebook is a tour of its main entry points.
It produces four example figures, all exported as **true vector PDF** in `figures/`:

1. **2D**: $y_1 = \sin x$ and $y_2 = \cos x$.
2. **3D helicoidal trajectory** at all three sizes (`full`, `half`, `third`).
3. **Sphere assembly** with body axes — the matplotlib-3D depth-ordering canary.
4. **Label add/remove/displace** demo (2D and 3D).

## Setup

In [ ]:
import numpy as np

from softmobility.classes import figstyle

figstyle.apply()  # set rcParams to the paper aesthetic
FIGDIR = "figures"

## 1. 2D demo

In [ ]:
x = np.linspace(0.0, 2 * np.pi, 200)
y1 = np.sin(x)
y2 = np.cos(x)

fig, ax = figstyle.figure(size="half", aspect=4 / 3)
ax.plot(x, y1, color=figstyle.COLORS["red"], linewidth=1.5, label="sin(x)")
ax.plot(x, y2, color=figstyle.COLORS["blue"], linewidth=1.5, label="cos(x)")
ax.set_xlim(0, 2 * np.pi)
ax.set_ylim(-1.1, 1.1)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend(loc="lower left", frameon=False)

figstyle.save(fig, "fig_2d_sin_cos", figdir=FIGDIR)

## 2. 3D helicoidal trajectory at full / half / third sizes

A helix $\mathbf{r}(t) = (\cos t,\; \sin t,\; t/3\pi)$ over $t \in [0, 6\pi]$, projected onto the three back walls of the bounding cube and rendered at every named size.
Every PDF is fully vector.

In [ ]:
t = np.linspace(0.0, 6 * np.pi, 600)
xs = np.cos(t)
ys = np.sin(t)
zs = t / (3 * np.pi)
bounds = ((-1.5, 1.5), (-1.5, 1.5), (-0.5, 2.5))

for size in ("full", "half", "third"):
    fig, ax = figstyle.figure_3d(size=size, aspect=1.0)
    figstyle.add_back_panels(ax, *bounds, color="black", width=1.0)
    figstyle.add_shadow(ax, xs, ys, zs, plane="xy_low", bounds=bounds)
    figstyle.add_shadow(ax, xs, ys, zs, plane="xz_low", bounds=bounds)
    figstyle.add_shadow(ax, xs, ys, zs, plane="yz_low", bounds=bounds)
    ax.plot(xs, ys, zs, color=figstyle.COLORS["red"], linewidth=1.5, label="helix")
    ax.set_xlim(*bounds[0])
    ax.set_ylim(*bounds[1])
    ax.set_zlim(*bounds[2])
    figstyle.save(fig, f"helix_{size}", figdir=FIGDIR)

## 3. Sphere assembly with body axes


In [ ]:
positions = np.array(
    [
        [0.0, 0.0, 0.0],
        [0.6, 0.0, 0.0],
        [0.0, 0.8, 0.0],
        [0.0, 0.0, 1.0],
    ]
)
radii = np.array([0.5, 0.1, 0.3, 0.5])

fig, ax = figstyle.figure_3d(size="half", aspect=1.0)

axis_length = 1.4
figstyle.add_body_axes(ax, length=axis_length, color=figstyle.COLORS["black"])

for centre, radius in zip(positions, radii, strict=True):
    figstyle.add_sphere(ax, centre, radius)

bound = float(max(radii.max(), axis_length * 1.2))
ax.set_xlim(-bound, bound)
ax.set_ylim(-bound, bound)
ax.set_zlim(-bound, bound)

figstyle.save(fig, "fig_sphere_assembly", figdir=FIGDIR)

## 4. Adding, removing, displacing labels


In [ ]:
fig_lbl, ax_lbl = figstyle.figure(size="half", aspect=4 / 3)
ax_lbl.plot(x, y1, color=figstyle.COLORS["red"], linewidth=1.5)
ax_lbl.plot(x, y2, color=figstyle.COLORS["blue"], linewidth=1.5)
ax_lbl.set_xlim(0, 2 * np.pi)
ax_lbl.set_ylim(-1.4, 1.4)
ax_lbl.set_xlabel("x")
ax_lbl.set_ylabel("y")

figstyle.add_label(
    ax_lbl,
    (np.pi / 2, 1.35),
    "max(sin)",
    color=figstyle.COLORS["red"],
    anchor="bottom center",
    offset=(0, 0.05),
    name="sin_max",
)
figstyle.add_label(
    ax_lbl,
    (np.pi, -1.35),
    "min(cos)",
    color=figstyle.COLORS["blue"],
    anchor="bottom left",
    offset=(0.05, 0.05),
    name="cos_max",
)

n_removed = figstyle.remove_label(ax_lbl, "cos_max")
n_moved = figstyle.displace_label(ax_lbl, "sin_max", offset=(0.4, -0.1))
print(f"removed {n_removed}, moved {n_moved}")

### Same helpers in 3D

In [ ]:
fig3, ax3 = figstyle.figure_3d(size="half", aspect=1.0)
figstyle.add_body_axes(ax3, length=1.4, color=figstyle.COLORS["black"], show_labels=True)
for centre, radius in zip(positions, radii, strict=True):
    figstyle.add_sphere(ax3, centre, radius)
ax3.scatter([0], [0], [0], color=figstyle.COLORS["black"], marker="x", s=30)

figstyle.displace_label(ax3, "axis_label_E1", text="X", offset=(0, 0, 0.2))
figstyle.remove_label(ax3, "axis_label_E2")
figstyle.displace_label(ax3, "axis_label_E3", new_position=(0.5, 0.0, 1.75))
figstyle.add_label(
    ax3,
    (0.6, 0, -0.2),
    "sphere 1",
    color=figstyle.COLORS["red"],
    anchor="bottom center",
    offset=(0, 0, -0.2),
    name="sphere1_caption",
)

bound = float(max(radii.max(), 1.4 * 1.2))
ax3.set_xlim(-bound, bound)
ax3.set_ylim(-bound, bound)
ax3.set_zlim(-bound, bound)